In [73]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option("display.expand_frame_repr", False)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [74]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [75]:
from google.colab import drive
drive.mount('/content/drive/')

DATA_DIR = "/content/drive/MyDrive/CS 489" 
FILE = os.path.join(DATA_DIR, "CS 489 - Self Playing - Data Collection (1).csv")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:

# Separate dataset into 75/10/15 split.
TRAIN_SPLIT = 0.75 
VAL_SPLIT   = 0.10
TEST_SPLIT  = 0.15
df          = pd.read_csv(FILE)
train_parts, val_parts, test_parts = [], [], []

# Scan through all unique word pairs (30).
for pair_id, group in df.groupby("anchor_left (0)"):
    
    # Extract 15% of total dataset into test set. 
    train_and_val, test = train_test_split(
        group, test_size = TEST_SPLIT, random_state = RANDOM_SEED)
    
    # Extract 10% of remaining 85% for validation split.
    train, val = train_test_split(
        train_and_val, test_size = VAL_SPLIT / (1 - TEST_SPLIT), random_state = RANDOM_SEED)

    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

train_df = pd.concat(train_parts).reset_index(drop = True)
val_df   = pd.concat(val_parts).reset_index(drop = True)
test_df  = pd.concat(test_parts).reset_index(drop = True)

print(train_df)
print(val_df)
print(test_df)


         anchor_left (0)  anchor_right (100)                     clue  target response_list response_mean  nathan kim  dp    category difference
0                    Bad                Good          Cutting in line    34.0            33            33         NaN NaN  Subjective          1
1                    Bad                Good           Curing disease    91.0            92            92         NaN NaN  Subjective          1
2                    Bad                Good       Fostering children    70.0            90            90         NaN NaN  Subjective         20
3                    Bad                Good        Community service    76.0            75            75         NaN NaN  Subjective          1
4                    Bad                Good         Selfless heroism   100.0            90            90         NaN NaN  Subjective         10
5                    Bad                Good     Forgetting birthdays    34.0            23            23         NaN NaN  Subject

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

clues = ["fire", "Freezing"]
pairs = ["lava", "Freeze"]

clue_embeddings = model.encode(clues)
pairs_embeddings = model.encode(pairs)

print(cosine_similarity(clue_embeddings[0], pairs_embeddings[0]))
print(cosine_similarity(clue_embeddings[1], pairs_embeddings[1]))


0.4812536
0.8812692
